In [1]:
# Setup: run this first.
# In Databricks, `spark` already exists and this cell only sets CATALOG.
# In VS Code, it connects to Databricks serverless through Databricks Connect.
import glob, os, shutil

CATALOG = "hertz"

try:
    spark
except NameError:
    # Login uses the `databricks` CLI. If it isn't on PATH, use the copy bundled with the VS Code extension.
    if not shutil.which("databricks"):
        bundled = sorted(glob.glob(os.path.expanduser(r"~\.vscode\extensions\databricks.databricks-*\bin")))
        if bundled:
            os.environ["PATH"] = bundled[-1] + os.pathsep + os.environ["PATH"]
    os.environ.setdefault("DATABRICKS_CONFIG_PROFILE", "vscode")

    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

print("Connected. Spark version:", spark.version)

Connected. Spark version: 4.2.0


# Bronze: load the raw CSVs into tables
Each CSV in `/Volumes/hertz/bronze/raw` becomes one `hertz.bronze.*` table, **unchanged**. Every column is kept as a string, and two lineage columns are added. Bronze is never cleaned, so silver can always be rebuilt from it.

In [0]:
from pyspark.sql import functions as F

CATALOG = "hertz"
VOLUME_PATH = f"/Volumes/{CATALOG}/bronze/raw"
DATASETS = ["job_types", "rental_demand", "vehicle_classes", "work_orders"]

for name in DATASETS:
    df = (spark.read
          .option("header", True)
          .option("inferSchema", False)          # bronze = raw strings; types are set in silver
          .csv(f"{VOLUME_PATH}/{name}.csv")
          .withColumn("_source_file", F.col("_metadata.file_path"))
          .withColumn("_ingested_at", F.current_timestamp()))

    (df.write.mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"{CATALOG}.bronze.{name}"))
    print(f"{CATALOG}.bronze.{name}: {df.count():,} rows")

hertz.bronze.job_types: 7 rows
hertz.bronze.rental_demand: 42 rows
hertz.bronze.vehicle_classes: 6 rows
hertz.bronze.work_orders: 65 rows


# Silver: clean each table
Reads from `hertz.bronze.*` and writes `hertz.silver.*` tables.

**Cleaning rules**
1. **Types**: dates, timestamps, numbers and booleans are cast explicitly, so nothing depends on schema inference.
2. **Categorical values** (`vehicle_class`, `job_type`, `status`) are trimmed and lowercased, so joins match across tables (`battery` = `Battery`). ID columns are left as-is.
3. **Exact duplicate rows** are dropped (WO-054, WO-024, WO-007).
4. **Open/Closed conflicts**: when the same vehicle, job and `opened_ts` appear as both `open` and `closed`, the job is done. The stale `open` row is dropped (WO-015, WO-031, WO-008, WO-058). *Assumption: a Closed record is the newer truth.*
5. **Closed rows are kept** in silver, so the history stays complete. The gold layer filters to `status = 'open'` for planning.

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")

def bronze(name):
    return spark.table(f"{CATALOG}.bronze.{name}")

def category(col):
    """Normalize a categorical column so it joins consistently across tables."""
    return F.lower(F.trim(F.col(col))).alias(col)

def save_silver(df, name):
    (df.write.mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"{CATALOG}.silver.{name}"))
    print(f"{CATALOG}.silver.{name}: {df.count():,} rows")

# --- job_types ---------------------------------------------------------------
job_types = bronze("job_types").select(
    category("job_type"),
    F.col("shop_hours").cast("double").alias("shop_hours"),      # NULL for vendor-only jobs
    F.col("parts_cost").cast("double").alias("parts_cost"),
    (F.upper(F.trim("in_house")) == "Y").alias("in_house"),     # Y/N -> boolean
    F.col("vendor_price").cast("double").alias("vendor_price"),
    F.col("vendor_days").cast("int").alias("vendor_days"),
).dropDuplicates()
save_silver(job_types, "job_types")

# --- vehicle_classes ---------------------------------------------------------
vehicle_classes = bronze("vehicle_classes").select(
    category("vehicle_class"),
    F.col("daily_rate").cast("double").alias("daily_rate"),
).dropDuplicates()
save_silver(vehicle_classes, "vehicle_classes")

# --- rental_demand -----------------------------------------------------------
rental_demand = bronze("rental_demand").select(
    F.to_date("date").alias("date"),
    category("vehicle_class"),
    F.col("reservations").cast("int").alias("reservations"),
    F.col("cars_ready").cast("int").alias("cars_ready"),
).dropDuplicates()
save_silver(rental_demand, "rental_demand")

# --- work_orders -------------------------------------------------------------
wo = bronze("work_orders").select(
    F.trim("work_order_id").alias("work_order_id"),
    F.trim("vehicle_id").alias("vehicle_id"),
    category("vehicle_class"),
    category("job_type"),
    F.to_timestamp(F.trim("opened_ts"), "yyyy-MM-dd HH:mm").alias("opened_ts"),
    category("status"),
)
n_raw = wo.count()

# Rule 3: exact duplicates
wo = wo.dropDuplicates()
n_dedup = wo.count()

# Rule 4: drop 'open' rows that have a matching 'closed' row for the same job
job_key = ["vehicle_id", "job_type", "opened_ts"]
closed_jobs = wo.filter("status = 'closed'").select(*job_key)
stale_open = (wo.filter("status = 'open'")
                .join(closed_jobs, job_key, "left_semi")
                .select("work_order_id"))
print("Stale open rows dropped:", sorted(r.work_order_id for r in stale_open.collect()))
work_orders = wo.join(stale_open, "work_order_id", "left_anti")

print(f"work_orders: {n_raw} raw -> {n_dedup} after dedup -> {work_orders.count()} after resolving open/closed conflicts")
save_silver(work_orders, "work_orders")

hertz.silver.job_types: 7 rows
hertz.silver.vehicle_classes: 6 rows
hertz.silver.rental_demand: 42 rows
Stale open rows dropped: ['WO-008', 'WO-015', 'WO-031', 'WO-058']
work_orders: 65 raw -> 62 after dedup -> 58 after resolving open/closed conflicts
hertz.silver.work_orders: 58 rows


# Reviewing the silver tables
Check the schemas, look for leftover spelling or formatting variants, and run the data quality checks.

In [0]:
cleaned_dfs = {name: spark.table(f"{CATALOG}.silver.{name}") for name in DATASETS}

for name, df in cleaned_dfs.items():
    print(f"\n=== {CATALOG}.silver.{name} ===")
    print(f"Rows: {df.count():,}  |  Columns: {len(df.columns)}")
    df.printSchema()
    display(df.limit(20))


=== hertz.silver.job_types ===
Rows: 7  |  Columns: 6
root
 |-- job_type: string (nullable = true)
 |-- shop_hours: double (nullable = true)
 |-- parts_cost: double (nullable = true)
 |-- in_house: boolean (nullable = true)
 |-- vendor_price: double (nullable = true)
 |-- vendor_days: integer (nullable = true)



job_type,shop_hours,parts_cost,in_house,vendor_price,vendor_days
oil change,0.75,45.0,true,195.0,1
collision repair,null,null,false,2260.0,5
tires,1.5,380.0,true,580.0,1
brakes,2.5,260.0,true,540.0,1
transmission,null,null,false,3260.0,7
ac repair,3.0,420.0,true,840.0,2
battery,1.5,210.0,true,400.0,1



=== hertz.silver.rental_demand ===
Rows: 42  |  Columns: 4
root
 |-- date: date (nullable = true)
 |-- vehicle_class: string (nullable = true)
 |-- reservations: integer (nullable = true)
 |-- cars_ready: integer (nullable = true)



date,vehicle_class,reservations,cars_ready
2026-10-18,economy,40,48
2026-10-12,full-size,15,19
2026-10-12,premium suv,10,10
2026-10-13,full-size,15,19
2026-10-14,full-size,15,19
2026-10-12,suv,24,24
2026-10-14,suv,40,24
2026-10-15,premium suv,15,10
2026-10-16,premium suv,15,10
2026-10-13,economy,40,48



=== hertz.silver.vehicle_classes ===
Rows: 6  |  Columns: 2
root
 |-- vehicle_class: string (nullable = true)
 |-- daily_rate: double (nullable = true)



vehicle_class,daily_rate
full-size,63.0
economy,42.0
premium suv,118.0
midsize,55.0
minivan,92.0
suv,82.0



=== hertz.silver.work_orders ===
Rows: 58  |  Columns: 6
root
 |-- work_order_id: string (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- vehicle_class: string (nullable = true)
 |-- job_type: string (nullable = true)
 |-- opened_ts: timestamp (nullable = true)
 |-- status: string (nullable = true)



work_order_id,vehicle_id,vehicle_class,job_type,opened_ts,status
WO-029,V-67249,suv,brakes,2026-10-10T22:00:00.000Z,open
WO-022,V-25061,full-size,tires,2026-10-09T12:00:00.000Z,open
WO-062,V-32363,midsize,collision repair,2026-10-08T22:00:00.000Z,closed
WO-047,V-50191,full-size,ac repair,2026-10-11T13:00:00.000Z,open
WO-050,V-28626,midsize,ac repair,2026-10-11T08:00:00.000Z,open
WO-028,V-22738,minivan,tires,2026-10-11T05:00:00.000Z,open
WO-057,V-47131,full-size,collision repair,2026-10-08T22:00:00.000Z,open
WO-054,V-27482,premium suv,ac repair,2026-10-09T20:00:00.000Z,open
WO-037,V-18833,suv,brakes,2026-10-10T16:00:00.000Z,open
WO-025,V-97072,full-size,tires,2026-10-08T17:00:00.000Z,open


In [0]:
# Check for spelling mistakes / inconsistent values in string columns
# Strategy: look at distinct values in low-cardinality string columns (likely
# categorical), and flag values that are suspiciously similar to each other
# (e.g. typos like "sedaan" vs "sedan", or casing/whitespace variants).

import difflib

for dataset_name, df in cleaned_dfs.items():
    print(f"\n==================== {dataset_name} ====================")

    string_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() == "string"]

    for c in string_cols:
        distinct_vals = [row[c] for row in df.select(c).distinct().collect() if row[c] is not None]

        # Skip free-text / high-cardinality columns; spelling review only
        # makes sense for a small set of category-like values
        if len(distinct_vals) == 0 or len(distinct_vals) > 30:
            continue

        print(f"\n-- {c} ({len(distinct_vals)} distinct values) --")
        for v in sorted(distinct_vals):
            count = df.filter(F.col(c) == v).count()
            print(f"   {v!r}: {count:,} rows")

        # Flag near-duplicate values that may be typos or formatting variants
        suspects = []
        for i, v1 in enumerate(distinct_vals):
            for v2 in distinct_vals[i + 1:]:
                s1, s2 = str(v1).strip().lower(), str(v2).strip().lower()
                ratio = difflib.SequenceMatcher(None, s1, s2).ratio()
                # Similar but not identical, and not just whitespace/casing copies
                if 0.75 <= ratio < 1.0:
                    suspects.append((v1, v2, ratio))
                # Whitespace or casing-only differences
                elif s1 == s2 and str(v1) != str(v2):
                    suspects.append((v1, v2, 1.0))

        if suspects:
            print("   >>> Possible spelling/formatting issues:")
            for v1, v2, ratio in suspects:
                print(f"       {v1!r}  <->  {v2!r}   (similarity: {ratio:.2f})")



==================== job_types ====================

-- job_type (7 distinct values) --
   'ac repair': 1 rows
   'battery': 1 rows
   'brakes': 1 rows
   'collision repair': 1 rows
   'oil change': 1 rows
   'tires': 1 rows
   'transmission': 1 rows

==================== rental_demand ====================

-- vehicle_class (6 distinct values) --
   'economy': 7 rows
   'full-size': 7 rows
   'midsize': 7 rows
   'minivan': 7 rows
   'premium suv': 7 rows
   'suv': 7 rows

==================== vehicle_classes ====================

-- vehicle_class (6 distinct values) --
   'economy': 1 rows
   'full-size': 1 rows
   'midsize': 1 rows
   'minivan': 1 rows
   'premium suv': 1 rows
   'suv': 1 rows

==================== work_orders ====================

-- vehicle_class (6 distinct values) --
   'economy': 12 rows
   'full-size': 11 rows
   'midsize': 12 rows
   'minivan': 2 rows
   'premium suv': 5 rows
   'suv': 16 rows

-- job_type (7 distinct values) --
   'ac repair': 10 rows
   'ba

In [0]:
# Data quality checks: stop the notebook if silver isn't fit for planning
wo, jt, vc, rd = (cleaned_dfs[n] for n in ["work_orders", "job_types", "vehicle_classes", "rental_demand"])

checks = {
    # every work order ID appears once
    "duplicate work_order_id":
        wo.groupBy("work_order_id").count().filter("count > 1").count(),
    # every job_type / vehicle_class in work orders exists in the lookup tables
    "work_orders.job_type not in job_types":
        wo.join(jt, "job_type", "left_anti").count(),
    "work_orders.vehicle_class not in vehicle_classes":
        wo.join(vc, "vehicle_class", "left_anti").count(),
    "rental_demand.vehicle_class not in vehicle_classes":
        rd.join(vc, "vehicle_class", "left_anti").count(),
    # casts that silently failed become NULL
    "work_orders with NULL opened_ts":
        wo.filter(F.col("opened_ts").isNull()).count(),
    "rental_demand with NULL date":
        rd.filter(F.col("date").isNull()).count(),
    # in-house jobs need shop hours and parts cost to be costed
    "in-house job_types missing shop_hours/parts_cost":
        jt.filter("in_house AND (shop_hours IS NULL OR parts_cost IS NULL)").count(),
}

for name, bad in checks.items():
    print(f"{'OK  ' if bad == 0 else 'FAIL'}  {name}: {bad}")

display(wo.groupBy("status").count())
assert all(v == 0 for v in checks.values()), "Data quality checks failed, see above"

OK    duplicate work_order_id: 0
OK    work_orders.job_type not in job_types: 0
OK    work_orders.vehicle_class not in vehicle_classes: 0
OK    rental_demand.vehicle_class not in vehicle_classes: 0
OK    work_orders with NULL opened_ts: 0
OK    rental_demand with NULL date: 0
OK    in-house job_types missing shop_hours/parts_cost: 0


status,count
open,54
closed,4


In [2]:
# The week at a glance: reserved vs. ready vs. in repair, per class
import pandas as pd

rd = spark.table(f"{CATALOG}.silver.rental_demand").toPandas()
vc = spark.table(f"{CATALOG}.silver.vehicle_classes").toPandas()
jobs = (spark.table(f"{CATALOG}.silver.work_orders").filter("status = 'open'")
        .join(spark.table(f"{CATALOG}.silver.job_types"), "job_type").toPandas())

repairs = (jobs.groupby("vehicle_class")
           .agg(in_repair=("work_order_id", "count"),
                shop_jobs=("in_house", "sum"))
           .assign(vendor_only=lambda d: d.in_repair - d.shop_jobs))

summary = (rd.assign(gap=rd.reservations - rd.cars_ready)
           .groupby("vehicle_class")
           .agg(cars_ready=("cars_ready", "max"),
                peak_reserved=("reservations", "max"),
                peak_short=("gap", "max"),
                days_short=("gap", lambda g: (g > 0).sum()))
           .join(repairs)
           .join(vc.set_index("vehicle_class"))
           .assign(peak_short=lambda d: d.peak_short.clip(lower=0),
                   short_if_all_fixed=lambda d: (d.peak_reserved - d.cars_ready - d.in_repair).clip(lower=0))
           .sort_values("peak_short", ascending=False)
           .reset_index())

display(summary[["vehicle_class", "daily_rate", "cars_ready", "peak_reserved", "peak_short", "days_short",
                 "in_repair", "shop_jobs", "vendor_only", "short_if_all_fixed"]])

# Day-by-day shortage before any repairs (reservations - cars ready, never below 0)
display(rd.assign(short=(rd.reservations - rd.cars_ready).clip(lower=0))
          .pivot(index="vehicle_class", columns="date", values="short")
          .loc[summary.vehicle_class])

,vehicle_class,daily_rate,cars_ready,peak_reserved,peak_short,days_short,in_repair,shop_jobs,vendor_only,short_if_all_fixed
0,suv,82.0,24,40,16,6,15,15,0,1
1,midsize,55.0,43,53,10,6,11,11,0,0
2,premium suv,118.0,10,15,5,6,5,5,0,0
3,minivan,92.0,13,15,2,6,2,2,0,0
4,full-size,63.0,19,15,0,0,10,8,2,0
5,economy,42.0,48,40,0,0,11,10,1,0


date,2026-10-12,2026-10-13,2026-10-14,2026-10-15,2026-10-16,2026-10-17,2026-10-18
vehicle_class,,,,,,,
suv,0,16,16,16,16,16,16
midsize,0,2,4,6,8,10,8
premium suv,0,5,5,5,5,5,5
minivan,0,2,2,2,2,2,2
full-size,0,0,0,0,0,0,0
economy,0,0,0,0,0,0,0


# Baseline: oldest first (FIFO)
How the shop works today, so we have a number to beat.

**How it works**
1. Load the open jobs from silver.
2. Put the in-house jobs in a queue, oldest `opened_ts` first.
3. Work the queue: 12 tech hours per day, Mon–Fri. A finished car is ready the next morning.
4. Vendor-only jobs (`in_house = false`) are sent to the vendor Monday. Ready = Monday + `vendor_days`.
5. Shortage per class per day = `reservations − cars_ready − cars back so far`, never below 0. Each short car costs one daily rate.
6. Week cost = shop + vendor + lost revenue.

**Assumptions to confirm** (not stated in the brief)
- **A.** A job can span two days. Unfinished work carries over to the next open day.
- **B.** Parts are charged when a job starts, even if it finishes after the week ends.
- **C.** In the baseline, vendor-only jobs are sent Monday, whatever their class.

In [3]:
import pandas as pd
from datetime import date, timedelta

# --- Settings: same values as config/planning.yaml ----------------------------
# Typed in here for now; later we'll read them from the config file instead.
WEEK_START, WEEK_END = date(2026, 10, 12), date(2026, 10, 18)
TECH_HOURS_PER_DAY   = 12
SHOP_OPEN_DAYS       = {"Mon", "Tue", "Wed", "Thu", "Fri"}
LABOR_RATE           = 68
SHOP_READY_DELAY     = timedelta(days=1)      # finished today -> ready tomorrow morning

# --- Step 1: load silver tables into pandas ----------------------------------
# The tables are tiny, and a day-by-day simulation is easier to read in plain Python than in Spark.
silver = lambda t: spark.table(f"{CATALOG}.silver.{t}").toPandas()
work_orders, job_types = silver("work_orders"), silver("job_types")
vehicle_classes, rental_demand = silver("vehicle_classes"), silver("rental_demand")

open_jobs = work_orders[work_orders.status == "open"].merge(job_types, on="job_type")
print(f"Step 1: {len(open_jobs)} open jobs: "
      f"{open_jobs.in_house.sum()} in-house, {(~open_jobs.in_house).sum()} vendor-only")

# --- Step 2: shop queue, oldest first -----------------------------------------
week      = [WEEK_START + timedelta(days=i) for i in range((WEEK_END - WEEK_START).days + 1)]
shop_days = [d for d in week if d.strftime("%a") in SHOP_OPEN_DAYS]

queue = (open_jobs[open_jobs.in_house]
         .sort_values(["opened_ts", "work_order_id"])     # ID breaks ties between equal timestamps
         .reset_index(drop=True))
print(f"Step 2: queue needs {queue.shop_hours.sum()} shop hours; "
      f"the week has {len(shop_days) * TECH_HOURS_PER_DAY} ({len(shop_days)} days x {TECH_HOURS_PER_DAY})")

# --- Step 3: work the queue day by day ----------------------------------------
day_i, hours_left = 0, TECH_HOURS_PER_DAY
starts, finishes, worked = [], [], []

for hours_needed in queue.shop_hours:
    remaining, start, finish = hours_needed, None, None
    while remaining > 1e-9 and day_i < len(shop_days):        # stop when job done or week over
        start = start or shop_days[day_i]
        work = min(remaining, hours_left)
        remaining  -= work
        hours_left -= work
        if remaining <= 1e-9:
            finish = shop_days[day_i]
        if hours_left <= 1e-9:                                  # day used up -> next open day (assumption A)
            day_i, hours_left = day_i + 1, TECH_HOURS_PER_DAY
    starts.append(start); finishes.append(finish); worked.append(hours_needed - remaining)

queue["start_day"]    = starts
queue["finish_day"]   = finishes
queue["hours_worked"] = worked
queue["ready_date"]   = [f + SHOP_READY_DELAY if f else None for f in finishes]
queue["shop_cost"]    = (queue.hours_worked * LABOR_RATE
                         + queue.parts_cost.where(queue.hours_worked > 0, 0))   # assumption B

queue.index = queue.index + 1                                  # queue position 1, 2, 3...
print(f"Step 3: {queue.finish_day.notna().sum()} jobs finished this week, "
      f"{queue.finish_day.isna().sum()} still waiting")
display(queue[["work_order_id", "vehicle_class", "job_type", "opened_ts", "shop_hours",
               "start_day", "finish_day", "ready_date", "shop_cost"]])

# --- Step 4: vendor-only jobs go out Monday (assumption C) --------------------
vendor = open_jobs[~open_jobs.in_house].copy()
vendor["sent_date"]  = WEEK_START
vendor["ready_date"] = [WEEK_START + timedelta(days=int(n)) for n in vendor.vendor_days]
print("Step 4: vendor jobs")
display(vendor[["work_order_id", "vehicle_class", "job_type", "vendor_price", "sent_date", "ready_date"]])

# --- Step 5: shortages per class per day --------------------------------------
cars_back = pd.concat([queue[["vehicle_class", "ready_date"]],
                       vendor[["vehicle_class", "ready_date"]]]).dropna()

demand = (rental_demand[(rental_demand.date >= WEEK_START) & (rental_demand.date <= WEEK_END)]
          .merge(vehicle_classes, on="vehicle_class"))
demand["cars_back"] = [((cars_back.vehicle_class == c) & (cars_back.ready_date <= d)).sum()
                       for c, d in zip(demand.vehicle_class, demand.date)]
demand["short"]        = (demand.reservations - demand.cars_ready - demand.cars_back).clip(lower=0)
demand["lost_revenue"] = demand.short * demand.daily_rate
print("Step 5: cars short per class per day")
display(demand.pivot(index="vehicle_class", columns="date", values="short"))

# --- Step 6: cost of the week -------------------------------------------------
shop_total, vendor_total, lost_total = queue.shop_cost.sum(), vendor.vendor_price.sum(), demand.lost_revenue.sum()
print("Step 6: cost of the week (baseline, oldest first)")
print(f"  Shop cost     ${shop_total:>10,.0f}")
print(f"  Vendor cost   ${vendor_total:>10,.0f}")
print(f"  Lost revenue  ${lost_total:>10,.0f}")
print(f"  TOTAL         ${shop_total + vendor_total + lost_total:>10,.0f}")

Step 1: 54 open jobs: 51 in-house, 3 vendor-only
Step 2: queue needs 89.0 shop hours; the week has 60 (5 days x 12)
Step 3: 35 jobs finished this week, 16 still waiting


,work_order_id,vehicle_class,job_type,opened_ts,shop_hours,start_day,finish_day,ready_date,shop_cost
1,WO-021,economy,tires,2026-10-08 00:00:00,1.50,2026-10-12,2026-10-12,2026-10-13,482.0
2,WO-044,premium suv,battery,2026-10-08 00:00:00,1.50,2026-10-12,2026-10-12,2026-10-13,312.0
3,WO-052,full-size,ac repair,2026-10-08 02:00:00,3.00,2026-10-12,2026-10-12,2026-10-13,624.0
4,WO-016,midsize,oil change,2026-10-08 05:00:00,0.75,2026-10-12,2026-10-12,2026-10-13,96.0
5,WO-014,minivan,oil change,2026-10-08 07:00:00,0.75,2026-10-12,2026-10-12,2026-10-13,96.0
6,WO-030,full-size,brakes,2026-10-08 08:00:00,2.50,2026-10-12,2026-10-12,2026-10-13,430.0
7,WO-048,midsize,ac repair,2026-10-08 15:00:00,3.00,2026-10-12,2026-10-13,2026-10-14,624.0
8,WO-039,full-size,battery,2026-10-08 16:00:00,1.50,2026-10-13,2026-10-13,2026-10-14,312.0
9,WO-025,full-size,tires,2026-10-08 17:00:00,1.50,2026-10-13,2026-10-13,2026-10-14,482.0
10,WO-018,economy,tires,2026-10-08 18:00:00,1.50,2026-10-13,2026-10-13,2026-10-14,482.0


Step 4: vendor jobs


,work_order_id,vehicle_class,job_type,vendor_price,sent_date,ready_date
5,WO-057,full-size,collision repair,2260.0,2026-10-12,2026-10-17
27,WO-055,full-size,transmission,3260.0,2026-10-12,2026-10-19
31,WO-056,economy,transmission,3260.0,2026-10-12,2026-10-19


Step 5: cars short per class per day


date,2026-10-12,2026-10-13,2026-10-14,2026-10-15,2026-10-16,2026-10-17,2026-10-18
vehicle_class,,,,,,,
economy,0,0,0,0,0,0,0
full-size,0,0,0,0,0,0,0
midsize,0,1,1,2,2,3,1
minivan,0,1,1,1,1,1,1
premium suv,0,4,4,3,2,2,2
suv,0,16,13,11,10,7,7


Step 6: cost of the week (baseline, oldest first)
  Shop cost     $    12,685
  Vendor cost   $     8,780
  Lost revenue  $     8,356
  TOTAL         $    29,821
